In [ ]:
# Kaggle: enable Internet in the notebook settings, then run once.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pymoo', 'opacus'], check=False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 2.4 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'pymoo', 'opacus'], returncode=0)

In [ ]:
import os, json, time, math, glob, gc, random, warnings, itertools, copy, hashlib
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call, vmap, grad
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')

CFG = dict(
    # ---------------- paths (dhoogla parquet datasets; auto-discovered under /kaggle/input if moved) ----------------
    CIC2018_DIR='/kaggle/input/datasets/dhoogla/csecicids2018',     # 10 day files (main dataset)
    CIC2017_DIR='/kaggle/input/datasets/dhoogla/cicids2017',        # optional extra cross-dataset test
    UNSW_TRAIN='/kaggle/input/datasets/dhoogla/unswnb15/UNSW_NB15_training-set.parquet',
    UNSW_TEST='/kaggle/input/datasets/dhoogla/unswnb15/UNSW_NB15_testing-set.parquet',
    OUT='/kaggle/working/nexus_control',
    # ---------------- reproducibility ----------------
    SEED=42, SEEDS=[42, 43, 44],                             # multi-seed runs (ablation, Byzantine)
    # ---------------- data ----------------
    CHUNK=500_000,                                           # rows per CSV chunk
    BENIGN_KEEP_FRAC=0.05,                                   # benign rows kept while reading (memory)
    MAX_PER_CLASS_PER_DAY=60_000,                            # cap per (day, class) after reading
    MIN_CLASS_TOTAL=200,                                     # classes smaller than this are reported, then dropped
    DROP_DST_PORT=True,                                      # Dst Port used only for graph edges, not as a feature
    VAR_THRESHOLD=1e-3, CORR_THRESHOLD=0.95,
    # ---------------- leakage-resistant split ----------------
    WINDOW_SEC=30,                                           # tumbling graph window
    BLOCK_MIN=2,                                             # contiguous time block (assigned as a unit)
    SPLIT_FRACS=(0.70, 0.15, 0.15),                          # share of each class's blocks -> train/val/test
    ROWS_PER_SEC_PROXY=20,                                   # only if a file has no Timestamp: pseudo-time = row position / this
    REMOVE_CROSS_NEAR_DUP=True,                              # drop test flows that near-duplicate a training flow
    # ---------------- model ----------------
    MAX_NODES=64, KNN=5, SCALES=[1, 5, 15, 30],
    HIDDEN=128, TOKENS=8, N_LAYERS=6, N_RULES=20, DROPOUT=0.2, HEADS=4,
    OPS=['identity', 'linear', 'conv3', 'conv5', 'dilconv3', 'attention', 'graphconv', 'maxpool'],
    # ---------------- training ----------------
    BATCH_WINDOWS=32, EPOCHS=15, LR=1e-3, WD=1e-4, PATIENCE=5, RULE_L1=1e-3, AUX_RULE_W=0.5, RULE_LR_MULT=10,
    # ---------------- DARTS ----------------
    DARTS_EPOCHS=10, DARTS_ALPHA_LR=3e-4, DARTS_MAX_WINDOWS=6000,
    # ---------------- NSGA-III ----------------
    NSGA_POP=20, NSGA_GEN=10, NSGA_PARTITIONS=2, NSGA_EVAL_EPOCHS=2, NSGA_MAX_WINDOWS=4000,
    WIDTHS=[64, 128, 256], RULE_OPTIONS=[10, 20, 30], DROPOUTS=[0.1, 0.2, 0.3],
    FGSM_EPS=0.1,
    # ---------------- statistics ----------------
    BOOTSTRAP=1000, ECE_BINS=15, THRESHOLDS=[0.35, 0.50, 0.65],
    CV_FOLDS=5, CV_REPEATS=2, CV_EPOCHS=8,
    # ---------------- federated / DP ----------------
    FL_CLIENTS=10, FL_ROUNDS=50, FL_LOCAL_EPOCHS=3,          # main FedIDS configuration (as in the paper)
    FL_SWEEP_ROUNDS=20, FL_SWEEP_LOCAL_EPOCHS=1,             # cheaper settings for sweeps (reported as such)
    DP_CLIP=1.0, DP_Q=0.01, DP_SIGMA=1.1, DP_DELTA=1e-5,
    EPS_GRID=[0.5, 1.0, 2.0, 4.0, 8.0], DIRICHLET=[1.0, 0.5, 0.1],
    FL_LR=1e-3,
    # ---------------- Byzantine ----------------
    BYZ_ROUNDS=20, BYZ_LOCAL_EPOCHS=1, BYZ_RATIOS=[0.0, 0.1, 0.2, 0.3, 0.4],
    BYZ_ATTACKS=['sign_flip', 'gaussian', 'label_flip', 'scaling'],
    AGGREGATORS=['fedavg', 'median', 'krum', 'trimmed_mean'], TRIM_BETA=0.1,
    BYZ_CLIENT_MAX_WINDOWS=500,                              # windows sampled per client per local epoch (speed)
    # ---------------- other ----------------
    LODO_EPOCHS=5, SCAL_FRACS=[0.01, 0.05, 0.1, 0.25, 0.5, 1.0], SCAL_EPOCHS=5,
    UNSW_EPOCHS=15, CIC2017_EPOCHS=10,
    QUICK=False,
    RUN=dict(data=True, search=False, final=False, ablation=False, cv=False, leakage=False,
             fl=False, byzantine=False, explain=False, scalability=False, unsw=False, cic2017=False),
)
for _k in ['CIC2018_DIR', 'CIC2017_DIR', 'UNSW_TRAIN', 'UNSW_TEST', 'OUT']:          # optional overrides (testing)
    if os.environ.get('NEXUS_' + _k): CFG[_k] = os.environ['NEXUS_' + _k]
if os.environ.get('NEXUS_QUICK') == '1': CFG['QUICK'] = True
if CFG['QUICK']:   # smoke-test settings
    CFG.update(EPOCHS=2, DARTS_EPOCHS=1, NSGA_POP=8, NSGA_GEN=2, NSGA_EVAL_EPOCHS=1, NSGA_MAX_WINDOWS=300,
               DARTS_MAX_WINDOWS=300, NSGA_PARTITIONS=1, CV_FOLDS=3, CV_REPEATS=1, CV_EPOCHS=1, FL_ROUNDS=2, FL_LOCAL_EPOCHS=1,
               FL_SWEEP_ROUNDS=2, BYZ_ROUNDS=2, SEEDS=[42, 43], LODO_EPOCHS=1, SCAL_EPOCHS=1,
               SCAL_FRACS=[0.25, 1.0], UNSW_EPOCHS=1, CIC2017_EPOCHS=1, BOOTSTRAP=50, EPS_GRID=[1.0, 8.0], DIRICHLET=[0.5],
               BYZ_RATIOS=[0.0, 0.2], BYZ_ATTACKS=['sign_flip', 'label_flip'], MAX_PER_CLASS_PER_DAY=3000)

OUT = CFG['OUT']; os.makedirs(OUT, exist_ok=True)
for sub in ['figures', 'tables', 'raw', 'models']:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| torch', torch.__version__)

Device: cuda | torch 2.11.0+cu128


In [ ]:
# ---------------- utilities ----------------
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

set_seed(CFG['SEED'])
plt.rcParams.update({'font.size': 15, 'axes.titlesize': 16, 'axes.labelsize': 16, 'xtick.labelsize': 14,
                     'ytick.labelsize': 14, 'legend.fontsize': 13, 'figure.dpi': 110, 'savefig.dpi': 300})

TABLES = {}          # name -> DataFrame (all written to one Excel file at the end)
NUMBERS = {}         # flat dict of key numbers for the paper

def save_table(df, name, show=True):
    df = df.copy(); TABLES[name] = df
    df.to_csv(os.path.join(OUT, 'tables', name + '.csv'), index=False)
    if show:
        print(f'\n=== {name} ===')
        with pd.option_context('display.max_columns', 50, 'display.width', 250):
            print(df.to_string(index=False) if len(df) <= 60 else df.head(60).to_string(index=False))
    return df

def save_fig(fig, name, data=None):
    fig.tight_layout(); fig.savefig(os.path.join(OUT, 'figures', name + '.png'), bbox_inches='tight')
    if data is not None: pd.DataFrame(data).to_csv(os.path.join(OUT, 'figures', name + '_data.csv'), index=False)
    plt.show(); plt.close(fig)

def num(key, value):
    NUMBERS[key] = value.item() if hasattr(value, 'item') else value
    return value

class Timer:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); print(f'\n▶ {self.name} ...'); return self
    def __exit__(self, *a): self.dt = time.time() - self.t; print(f'✓ {self.name} finished in {self.dt/60:.1f} min')

def find_path(path, pattern):
    """Return path if it exists, otherwise search /kaggle/input recursively (Kaggle mount paths change)."""
    if os.path.exists(path): return path
    hits = sorted(glob.glob(os.path.join('/kaggle/input', '**', pattern), recursive=True))
    if not hits: return path
    return os.path.dirname(hits[0]) if pattern.startswith('*') and not os.path.splitext(path)[1] else hits[0]

CFG['CIC2018_DIR'] = find_path(CFG['CIC2018_DIR'], '*TrafficForML_CICFlowMeter.parquet')
CFG['CIC2017_DIR'] = find_path(CFG['CIC2017_DIR'], '*-no-metadata.parquet')
CFG['UNSW_TRAIN'] = find_path(CFG['UNSW_TRAIN'], 'UNSW_NB15_training-set.*')
CFG['UNSW_TEST'] = find_path(CFG['UNSW_TEST'], 'UNSW_NB15_testing-set.*')
for k in ['CIC2018_DIR', 'CIC2017_DIR', 'UNSW_TRAIN', 'UNSW_TEST']: print(f'{k:12s} -> {CFG[k]}  (exists: {os.path.exists(CFG[k])})')
json.dump(CFG, open(os.path.join(OUT, 'config.json'), 'w'), indent=2)

CIC2018_DIR  -> /kaggle/input/datasets/dhoogla/csecicids2018  (exists: True)
CIC2017_DIR  -> /kaggle/input/datasets/dhoogla/cicids2017  (exists: True)
UNSW_TRAIN   -> /kaggle/input/datasets/dhoogla/unswnb15/UNSW_NB15_training-set.parquet  (exists: True)
UNSW_TEST    -> /kaggle/input/datasets/dhoogla/unswnb15/UNSW_NB15_testing-set.parquet  (exists: True)


## Stage 1 — Data loading with real class names, cleaning and duplicate audit (R3.1, R4.1, R6.10–12)
The per-day parquet files of CSE-CIC-IDS2018 (dhoogla) are used: the file name gives the **capture day**
(for leave-one-day-out) and the `Label` column the **attack names**. Fine-grained labels are grouped into
categories (mapping saved as a table). If a file has no `Timestamp` column, time is approximated by the row
position in the file (`time_mode` in `T_read_audit`) — report this in the paper.

In [ ]:
ID_COLS = ['Flow ID', 'Src IP', 'Src Port', 'Dst IP']

def map_label(s):
    s = str(s).strip().lower()
    if s in ('label', 'nan', ''): return None
    if s == 'benign': return 'Benign'
    if 'ddos' in s: return 'DDoS'
    if 'portscan' in s or 'port scan' in s: return 'PortScan'
    if 'heartbleed' in s: return 'Other'
    if 'dos' in s: return 'DoS'
    if 'ftp' in s or 'ssh' in s or 'patator' in s: return 'BruteForce'
    if 'bot' in s: return 'Bot'
    if 'infil' in s: return 'Infiltration'
    if 'web' in s or 'xss' in s or 'sql' in s: return 'WebAttack'
    return 'Other'

def day_from_name(path):
    """CSE-CIC-IDS2018: dd-mm-yyyy in the file name -> 'yyyy-mm-dd'; otherwise the file stem (e.g. CIC-IDS2017)."""
    import re
    m = re.search(r'(\d{2})-(\d{2})-(\d{4})', os.path.basename(path))
    return f'{m.group(3)}-{m.group(2)}-{m.group(1)}' if m else os.path.basename(path).split('.')[0]

def iter_chunks(path):
    if path.endswith('.parquet'):
        yield pd.read_parquet(path)
    else:
        for ch in pd.read_csv(path, chunksize=CFG['CHUNK'], low_memory=False): yield ch

def load_flows(data_dir):
    """Reads all parquet/CSV files of a CIC dataset. Benign rows are subsampled while reading; attacks are kept
    and later capped per (day, class). Time = Timestamp if present, otherwise row position in the file."""
    rng = np.random.default_rng(CFG['SEED'])
    files = sorted(glob.glob(os.path.join(data_dir, '*.parquet')) + glob.glob(os.path.join(data_dir, '*.csv')))
    assert files, f'No parquet/CSV files in {data_dir}'
    parts, audit = [], []
    for f in files:
        day = day_from_name(f); n_read = n_bad = 0; offset = 0; mode = None; has_port = False
        for ch in iter_chunks(f):
            ch = ch.reset_index(drop=True)
            ch.columns = [str(c).strip() for c in ch.columns]
            pos = np.arange(len(ch)) + offset; offset += len(ch)
            lab = ch['Label'].astype(str).str.strip()
            ok = lab.str.lower() != 'label'
            ch, lab, pos = ch[ok], lab[ok], pos[ok.values]
            n_read += len(ch)
            cat = lab.map(map_label)
            ben = (cat == 'Benign').values
            keep = cat.notna().values & ((~ben) | (rng.random(len(ch)) < CFG['BENIGN_KEEP_FRAC']))
            ch, lab, cat, pos = ch[keep].copy(), lab[keep], cat[keep], pos[keep]
            ch = ch.drop(columns=[c for c in ID_COLS if c in ch.columns])
            if 'Timestamp' in ch.columns:
                ts = pd.to_datetime(ch['Timestamp'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
                if ts.isna().mean() > 0.5: ts = pd.to_datetime(ch['Timestamp'], dayfirst=True, errors='coerce')
                ch['t'] = (ts - ts.dt.normalize()).dt.total_seconds().values; mode = 'timestamp'
                ch = ch.drop(columns=['Timestamp'])
            else:
                ch['t'] = pos / CFG['ROWS_PER_SEC_PROXY']; mode = 'file-order (no Timestamp column)'
            has_port = has_port or ('Dst Port' in ch.columns)
            ch['orig_label'] = lab.values; ch['cat'] = cat.values; ch['day'] = day
            feat = [c for c in ch.columns if c not in ('Label', 'orig_label', 'cat', 't', 'day')]
            ch[feat] = ch[feat].apply(pd.to_numeric, errors='coerce').astype('float32')
            bad = ch[feat].replace([np.inf, -np.inf], np.nan).isna().any(axis=1) | ch['t'].isna()
            n_bad += int(bad.sum())
            parts.append(ch[~bad].drop(columns=['Label']))
        audit.append(dict(file=os.path.basename(f), day=day, rows_read=n_read, rows_invalid_after_sampling=n_bad,
                          time_mode=mode, dst_port_available=has_port))
        print(f'  {os.path.basename(f)}: {n_read:,} rows, time = {mode}, Dst Port: {has_port}')
    df = pd.concat(parts, ignore_index=True); del parts; gc.collect()
    r = pd.Series(np.random.default_rng(CFG['SEED']).random(len(df)), index=df.index)
    keep = r.groupby([df['day'], df['cat']]).rank(method='first') <= CFG['MAX_PER_CLASS_PER_DAY']
    return df[keep].reset_index(drop=True), pd.DataFrame(audit)

def dedup_audit(df, feat_cols):
    h = pd.util.hash_pandas_object(df[feat_cols], index=False).values
    df = df.assign(_h=h)
    n0 = len(df)
    conflict_h = df.groupby('_h')['cat'].nunique()
    conflict_h = set(conflict_h[conflict_h > 1].index)
    n_conflict = int(df['_h'].isin(conflict_h).sum())
    df = df[~df['_h'].isin(conflict_h)]
    n_before = len(df)
    df = df.drop_duplicates(subset=['_h', 'cat'])
    n_dup = n_before - len(df)
    audit = pd.DataFrame([dict(rows_before=n0, label_conflicting_rows_removed=n_conflict,
                               exact_duplicates_removed=n_dup, rows_after=len(df))])
    return df.drop(columns=['_h']).reset_index(drop=True), audit

META = ['orig_label', 'cat', 't', 'day', 'dport', 'proto']

def clean_and_label(df, tag):
    """port/protocol columns, de-duplication, removal of tiny classes, class ids. Returns df, classes, features."""
    if 'Dst Port' in df.columns:
        df['dport'] = df['Dst Port'].fillna(-1).astype(np.int64)
        if CFG['DROP_DST_PORT']: df = df.drop(columns=['Dst Port'])
    else:
        df['dport'] = -1
    df['proto'] = df['Protocol'].fillna(-1).astype(np.int64) if 'Protocol' in df.columns else -1
    feats = [c for c in df.columns if c not in META]
    incomplete = [c for c in feats if df[c].isna().any()]          # columns missing in some files
    if incomplete:
        print('  dropping columns not present in every file:', incomplete); df = df.drop(columns=incomplete)
        feats = [c for c in feats if c not in incomplete]
    df, dup = dedup_audit(df, feats); save_table(dup, f'T_duplicate_audit_{tag}')
    sizes = df['cat'].value_counts()
    small = sizes[(sizes < CFG['MIN_CLASS_TOTAL']) | (sizes.index == 'Other')]
    if len(small):
        save_table(small.rename_axis('class').reset_index(name='rows_dropped'), f'T_dropped_classes_{tag}')
        df = df[~df['cat'].isin(small.index)].reset_index(drop=True)
    classes = ['Benign'] + sorted([c for c in df['cat'].unique() if c != 'Benign'])
    df['y'] = df['cat'].map({c: i for i, c in enumerate(classes)}).astype(np.int64)
    return df, classes, feats

if CFG['RUN']['data']:
    with Timer('Stage 1: loading CSE-CIC-IDS2018'):
        raw, read_audit = load_flows(CFG['CIC2018_DIR'])
        save_table(read_audit, 'T_read_audit')
        save_table(raw.groupby(['orig_label', 'cat']).size().reset_index(name='rows_after_sampling'), 'T_label_mapping')
        raw, CLASSES, FEATS_RAW = clean_and_label(raw, 'cic2018')
        TIME_MODE = ', '.join(sorted(read_audit['time_mode'].astype(str).unique()))
        per_day = raw.pivot_table(index='cat', columns='day', values='y', aggfunc='size', fill_value=0)
        per_day['Total'] = per_day.sum(axis=1)
        save_table(per_day.reset_index(), 'T_class_by_day')
        print('Classes:', CLASSES, '| features:', len(FEATS_RAW), '| rows:', len(raw), '| time:', TIME_MODE)
        num('n_rows_after_cleaning', len(raw)); num('classes', CLASSES); num('n_raw_features', len(FEATS_RAW)); num('time_mode', TIME_MODE)


▶ Stage 1: loading CSE-CIC-IDS2018 ...
  Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet: 771,587 rows, time = file-order (no Timestamp column), Dst Port: False
  Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter.parquet: 619,346 rows, time = file-order (no Timestamp column), Dst Port: False
  DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet: 954,846 rows, time = file-order (no Timestamp column), Dst Port: False
  DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet: 561,396 rows, time = file-order (no Timestamp column), Dst Port: False
  DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet: 794,812 rows, time = file-order (no Timestamp column), Dst Port: False
  DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet: 591,873 rows, time = file-order (no Timestamp column), Dst Port: False
  Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter.parquet: 456,873 rows, time = file-order (no Timestamp column), Dst Port: False
  Infil2-Thursday-0

## Stage 2 — Leakage-resistant split (R3.1, R4.3, R4.8, R5.4)
* Flows are assigned to tumbling **30-s windows**; windows are grouped into **10-min blocks** per day.
* Whole blocks are assigned to train/validation/test. Classes are processed rarest first: the unassigned
  blocks containing a class are shuffled and split 70/15/15, so every class appears in every partition while
  a window — and the attack episode around it — never spans two partitions.
* After splitting, test flows that near-duplicate a training flow are counted (and removed).
* The preprocessing (imputation, variance/correlation filters, z-score) is **fitted on training data only**.

In [ ]:
def add_time_keys(df):
    df = df.copy()
    df['win'] = (df['t'] // CFG['WINDOW_SEC']).astype(np.int64)
    df['block'] = df['day'] + '_' + (df['t'] // (CFG['BLOCK_MIN'] * 60)).astype(int).astype(str)
    return df

def _class_block_order(df):
    """presence of each class in each block; classes ordered rarest first (benign last)"""
    pres = df.groupby(['block', 'y']).size().unstack(fill_value=0) > 0
    order = df['y'].value_counts().sort_values().index.tolist()
    order = [c for c in order if c != 0] + [0]
    return pres, order

def block_split(df, seed, fracs=None):
    """Whole blocks go to one partition. Classes are processed from rarest to most frequent; the still-unassigned
    blocks containing a class are shuffled and split by `fracs`, so every class with >= 3 such blocks appears in
    train, validation and test, and no block (hence no graph window) spans two partitions."""
    fracs = fracs or CFG['SPLIT_FRACS']; rng = np.random.default_rng(seed); pres, order = _class_block_order(df); assign = {}
    for c in order:
        cand = [b for b in pres.index[pres[c].values] if b not in assign]
        b = rng.permutation(cand); n = len(b)
        n_te = 0 if fracs[2] == 0 else (max(1, int(round(fracs[2] * n))) if n >= 2 else 0)
        n_va = 0 if fracs[1] == 0 else (max(1, int(round(fracs[1] * n))) if n >= 3 else 0)
        for i, blk in enumerate(b):
            assign[blk] = 'test' if i < n_te else ('val' if i < n_te + n_va else 'train')
    return df['block'].map(assign).values.astype(object)

def block_kfold(df, k, seed):
    """k folds of whole blocks; blocks containing each class (rarest first) are dealt round-robin."""
    rng = np.random.default_rng(seed); pres, order = _class_block_order(df); fold = {}; start = 0
    for c in order:
        cand = [b for b in pres.index[pres[c].values] if b not in fold]
        for i, blk in enumerate(rng.permutation(cand)): fold[blk] = (start + i) % k
        start += len(cand)
    return df['block'].map(fold).values.astype(int)

def near_dup_hash(X):
    return pd.util.hash_pandas_object(pd.DataFrame(np.round(X.astype(np.float64), 4)), index=False).values

class Preprocessor:
    """Fitted on training rows only."""
    def fit(self, df, feats, y):
        X = df[feats].values.astype(np.float64)
        self.median = np.nanmedian(X, axis=0)
        var = np.nanvar(X, axis=0)
        keep = var >= CFG['VAR_THRESHOLD']
        f1 = [f for f, k in zip(feats, keep) if k]
        X1 = X[:, keep]
        mu, sd = X1.mean(0), X1.std(0) + 1e-8
        Z = (X1 - mu) / sd
        sub = np.random.default_rng(0).choice(len(Z), min(50_000, len(Z)), replace=False)
        mi = mutual_info_classif(Z[sub], y[sub], random_state=0)
        C = np.corrcoef(Z[sub].T)
        drop = set()
        for i in range(len(f1)):
            for j in range(i + 1, len(f1)):
                if i in drop or j in drop: continue
                if abs(C[i, j]) > CFG['CORR_THRESHOLD']:
                    drop.add(j if mi[j] < mi[i] else i)
        idx = [i for i in range(len(f1)) if i not in drop]
        self.features = [f1[i] for i in idx]
        self.mean, self.std = mu[idx], sd[idx]
        self.var_removed = [f for f, k in zip(feats, keep) if not k]
        self.corr_removed = [f1[i] for i in sorted(drop)]
        return self
    def transform(self, df):
        X = df[self.features].values.astype(np.float64)
        return np.clip((X - self.mean) / self.std, -10, 10).astype(np.float32)

class WindowSet:
    """Flows grouped into graph windows (rows sorted by day/time); windows longer than MAX_NODES are chunked."""
    def __init__(self, X, y, t, dport, proto, group_keys, max_nodes=None, raw_index=None):
        max_nodes = max_nodes or CFG['MAX_NODES']
        order = np.lexsort((t, group_keys))
        self.X, self.y, self.t = X[order], y[order], t[order].astype(np.float32)
        self.dport, self.proto = dport[order], proto[order]
        self.raw_index = (raw_index[order] if raw_index is not None else order)
        g = group_keys[order]
        starts = np.r_[0, np.flatnonzero(g[1:] != g[:-1]) + 1]; ends = np.r_[starts[1:], len(g)]
        self.win = []
        for s, e in zip(starts, ends):
            for a in range(s, e, max_nodes): self.win.append((a, min(a + max_nodes, e)))
        self.win = np.array(self.win, dtype=np.int64)
    def __len__(self): return len(self.win)
    @property
    def n_flows(self): return len(self.y)
    def subset(self, widx):
        ws = copy.copy(self); ws.win = self.win[np.asarray(widx)]; return ws
    def window_labels(self):
        """dominant label per window (most frequent non-benign label, else benign)"""
        out = np.zeros(len(self.win), dtype=np.int64)
        for i, (s, e) in enumerate(self.win):
            yy = self.y[s:e]; att = yy[yy != 0]
            out[i] = np.bincount(att).argmax() if len(att) else 0
        return out
    def batches(self, bs, shuffle, rng=None):
        idx = np.arange(len(self.win))
        if shuffle: (rng or np.random).shuffle(idx)
        for b in range(0, len(idx), bs): yield self.collate(idx[b:b + bs])
    def collate(self, widx):
        w = self.win[widx]; L = (w[:, 1] - w[:, 0]); n = int(L.max()); B = len(w); d = self.X.shape[1]
        X = np.zeros((B, n, d), np.float32); Y = np.full((B, n), -1, np.int64); M = np.zeros((B, n), bool)
        T = np.zeros((B, n), np.float32); P = np.full((B, n), -1, np.int64); R = np.full((B, n), -2, np.int64)
        for i, (s, e) in enumerate(w):
            k = e - s; X[i, :k] = self.X[s:e]; Y[i, :k] = self.y[s:e]; M[i, :k] = True
            T[i, :k] = self.t[s:e] - self.t[s]; P[i, :k] = self.dport[s:e]; R[i, :k] = self.proto[s:e]
        tt = lambda a: torch.from_numpy(a).to(DEVICE)
        return dict(x=tt(X), y=tt(Y), mask=tt(M), t=tt(T), port=tt(P), proto=tt(R))

def make_windowset(df, X, raw_index=None):
    keys = (df['day'].astype(str) + '_' + df['win'].astype(str)).values
    keys = pd.factorize(keys)[0]
    return WindowSet(X, df['y'].values, df['t'].values, df['dport'].values, df['proto'].values, keys,
                     raw_index=raw_index)

def prepare_split(df, part, tag='main', remove_near_dup=None, feats=None):
    feats = feats or FEATS_RAW
    remove_near_dup = CFG['REMOVE_CROSS_NEAR_DUP'] if remove_near_dup is None else remove_near_dup
    tr, va, te = df[part == 'train'], df[part == 'val'], df[part == 'test']
    pre = Preprocessor().fit(tr, feats, tr['y'].values)
    Xtr, Xva, Xte = pre.transform(tr), pre.transform(va), pre.transform(te)
    htr = set(near_dup_hash(tr[feats].values))
    nd = np.array([h in htr for h in near_dup_hash(te[feats].values)], dtype=bool)
    info = dict(split=tag, train_flows=len(tr), val_flows=len(va), test_flows=len(te),
                test_near_duplicates_of_train=int(nd.sum()), removed=bool(remove_near_dup and nd.any()),
                shared_blocks_train_test=len(set(tr['block']) & set(te['block'])))
    if remove_near_dup and nd.any():
        te, Xte = te[~nd], Xte[~nd]
        info['test_flows_after_removal'] = len(te)
    S = dict(pre=pre, info=info,
             train=make_windowset(tr, Xtr, tr.index.values), val=make_windowset(va, Xva, va.index.values),
             test=make_windowset(te, Xte, te.index.values))
    return S

if CFG['RUN']['data']:
    with Timer('Stage 2: leakage-resistant split'):
        raw = add_time_keys(raw)
        PART = block_split(raw, CFG['SEED'])
        MAIN = prepare_split(raw, PART, 'time-block (main)')
        save_table(pd.DataFrame([MAIN['info']]), 'T_split_leakage_audit')
        cnt = pd.crosstab(raw['cat'], PART).reindex(index=CLASSES)[['train', 'val', 'test']]
        if MAIN['info'].get('removed'):
            cnt['test'] = pd.Series(MAIN['test'].y).map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES).fillna(0).astype(int)
        cnt['test_share_%'] = (100 * cnt['test'] / cnt['test'].sum()).round(2)
        save_table(cnt.reset_index().rename(columns={'cat': 'class'}), 'T_class_distribution_splits')
        pre = MAIN['pre']
        save_table(pd.DataFrame(dict(step=['input features', 'removed: variance < %g' % CFG['VAR_THRESHOLD'],
                                           'removed: |corr| > %g' % CFG['CORR_THRESHOLD'], 'retained'],
                                     n=[len(FEATS_RAW), len(pre.var_removed), len(pre.corr_removed), len(pre.features)],
                                     names=['', ', '.join(pre.var_removed), ', '.join(pre.corr_removed), ', '.join(pre.features)])),
                   'T_feature_selection')
        N_FEAT, N_CLS = len(pre.features), len(CLASSES)
        print(f'windows: train {len(MAIN["train"])}, val {len(MAIN["val"])}, test {len(MAIN["test"])}; features {N_FEAT}')
        num('n_features', N_FEAT); num('split_info', MAIN['info'])


▶ Stage 2: leakage-resistant split ...

=== T_split_leakage_audit ===
            split  train_flows  val_flows  test_flows  test_near_duplicates_of_train  removed  shared_blocks_train_test  test_flows_after_removal
time-block (main)       503919     103312      105100                             11     True                         0                    105089

=== T_class_distribution_splits ===
       class  train   val  test  test_share_%
      Benign 178975 38182 38738         36.86
         Bot  41531  7998 10434          9.93
  BruteForce  41956  9000  9028          8.59
        DDoS  84707 17043 18240         17.36
         DoS  87588 11790 11669         11.10
Infiltration  68826 19298 16463         15.67
   WebAttack    336     1   517          0.49

=== T_feature_selection ===
                     step  n                                                                                                                                                                               

## Model components
* **AutoIDS-NAS encoder** — each flow's features are projected to `TOKENS` tokens × `C` channels; each of
  the 6 searchable layers is a DARTS mixed operation over 8 candidate operations
  (identity, linear, conv3, conv5, dilated conv3, token self-attention, token graph convolution, max-pool).
* **DSTGN** — nodes are the flows of one 30-s window; edges connect flows sharing a destination port and
  the k nearest neighbours in embedding space. Multi-scale temporal context (1, 5, 15, 30 s) is combined
  with a learned **scale attention** (these weights are reported), followed by two masked graph-attention layers.
* **NSAR** — differentiable fuzzy rules on the input features. Rule activation = selection-weighted geometric mean
  of sigmoid memberships (soft AND); logits fused as softmax(γ·z_N + (1−γ)·z_S). An auxiliary loss on z_S makes
  the rule pathway predictive on its own, so the extracted rules are faithful (fidelity is reported).
* All normalisation layers are LayerNorm (required for per-sample DP-SGD); attention is written explicitly
  so that the model works with `torch.func.vmap`.

In [ ]:
class TokOp(nn.Module):
    """Operation acting on x of shape [B, L, C]."""
    def __init__(self, kind, C, L, heads=4):
        super().__init__(); self.kind = kind
        if kind == 'linear': self.f = nn.Linear(C, C)
        elif kind in ('conv3', 'conv5', 'dilconv3'):
            k, d = {'conv3': (3, 1), 'conv5': (5, 1), 'dilconv3': (3, 2)}[kind]
            self.f = nn.Conv1d(C, C, k, padding=d * (k - 1) // 2, dilation=d)
        elif kind == 'attention':
            self.h = heads if C % heads == 0 else 1
            self.qkv = nn.Linear(C, 3 * C); self.o = nn.Linear(C, C)
        elif kind == 'graphconv':
            self.E = nn.Parameter(torch.randn(L, 8) * 0.1); self.f = nn.Linear(C, C)
        elif kind == 'maxpool': self.f = nn.MaxPool1d(3, 1, 1)
    def forward(self, x):
        k = self.kind
        if k == 'identity': return x
        if k == 'linear': return F.gelu(self.f(x))
        if k in ('conv3', 'conv5', 'dilconv3'): return F.gelu(self.f(x.transpose(1, 2)).transpose(1, 2))
        if k == 'maxpool': return self.f(x.transpose(1, 2)).transpose(1, 2)
        if k == 'graphconv':
            A = torch.softmax(self.E @ self.E.T, dim=-1)
            return F.gelu(self.f(torch.einsum('ij,bjc->bic', A, x)))
        if k == 'attention':
            B, L, C = x.shape; h = self.h
            q, kk, v = self.qkv(x).reshape(B, L, 3, h, C // h).permute(2, 0, 3, 1, 4)
            a = torch.softmax(q @ kk.transpose(-1, -2) / math.sqrt(C // h), dim=-1)
            return self.o((a @ v).transpose(1, 2).reshape(B, L, C))
        raise ValueError(k)

class SearchLayer(nn.Module):
    def __init__(self, C, L, ops, fixed=None):
        super().__init__(); self.fixed = fixed
        names = [fixed] if fixed is not None else ops
        self.ops = nn.ModuleList([TokOp(o, C, L) for o in names]); self.norm = nn.LayerNorm(C)
    def forward(self, x, alpha=None):
        z = self.norm(x)
        if self.fixed is not None: return x + self.ops[0](z)
        w = torch.softmax(alpha, dim=-1)
        return x + sum(wi * op(z) for wi, op in zip(w, self.ops))

class Encoder(nn.Module):
    """genotype=None -> DARTS supernet (mixed ops); list of op names -> discrete architecture; 'mlp' -> MLP."""
    def __init__(self, d, C, L, n_layers, genotype=None, dropout=0.2):
        super().__init__(); self.C, self.L, self.genotype = C, L, genotype
        if genotype == 'mlp':
            self.net = nn.Sequential(nn.Linear(d, C), nn.GELU(), nn.LayerNorm(C), nn.Dropout(dropout),
                                     nn.Linear(C, C), nn.GELU(), nn.LayerNorm(C)); return
        self.inp = nn.Linear(d, C * L)
        if genotype is None:
            self.alpha = nn.Parameter(1e-3 * torch.randn(n_layers, len(CFG['OPS'])))
            self.layers = nn.ModuleList([SearchLayer(C, L, CFG['OPS']) for _ in range(n_layers)])
        else:
            self.layers = nn.ModuleList([SearchLayer(C, L, None, fixed=g) for g in genotype])
        self.drop = nn.Dropout(dropout); self.out = nn.LayerNorm(C)
    def forward(self, x):                      # x: [M, d]
        if self.genotype == 'mlp': return self.net(x)
        z = self.inp(x).reshape(x.shape[0], self.L, self.C)
        for i, layer in enumerate(self.layers):
            z = layer(z, self.alpha[i] if self.genotype is None else None)
        return self.out(self.drop(z.mean(1)))

class MaskedGAT(nn.Module):
    def __init__(self, C, heads, dropout):
        super().__init__(); self.h = heads if C % heads == 0 else 1
        self.n1, self.n2 = nn.LayerNorm(C), nn.LayerNorm(C)
        self.qkv, self.o = nn.Linear(C, 3 * C), nn.Linear(C, C)
        self.ff = nn.Sequential(nn.Linear(C, 2 * C), nn.GELU(), nn.Linear(2 * C, C)); self.drop = nn.Dropout(dropout)
    def forward(self, x, adj):                 # x [B,N,C], adj [B,N,N] bool
        B, N, C = x.shape; h = self.h
        q, k, v = self.qkv(self.n1(x)).reshape(B, N, 3, h, C // h).permute(2, 0, 3, 1, 4)
        s = (q @ k.transpose(-1, -2)) / math.sqrt(C // h)
        s = s.masked_fill(~adj.unsqueeze(1), -1e9)
        a = self.drop(torch.softmax(s, dim=-1))
        x = x + self.o((a @ v).transpose(1, 2).reshape(B, N, C))
        return x + self.drop(self.ff(self.n2(x)))

class DSTGN(nn.Module):
    def __init__(self, C, heads, dropout, scales):
        super().__init__(); self.scales = scales
        self.wh, self.wm, self.v = nn.Linear(C, C), nn.Linear(C, C), nn.Linear(C, 1, bias=False)
        self.ctx, self.norm = nn.Linear(C, C), nn.LayerNorm(C)
        self.g1, self.g2 = MaskedGAT(C, heads, dropout), MaskedGAT(C, heads, dropout)
    def forward(self, h, t, mask, port):
        B, N, C = h.shape
        valid = mask.unsqueeze(1) & mask.unsqueeze(2)                       # [B,N,N]
        dt = (t.unsqueeze(2) - t.unsqueeze(1)).abs()
        ms = []
        for s in self.scales:
            Ms = (dt <= s / 2.0) & valid
            ms.append((Ms.float() @ h) / Ms.float().sum(-1, keepdim=True).clamp(min=1.0))
        m = torch.stack(ms, dim=2)                                         # [B,N,S,C]
        e = self.v(torch.tanh(self.wh(h).unsqueeze(2) + self.wm(m))).squeeze(-1)   # [B,N,S]
        a = torch.softmax(e, dim=-1)
        h = self.norm(h + self.ctx((a.unsqueeze(-1) * m).sum(2)))
        eye = torch.eye(N, dtype=torch.bool, device=h.device).unsqueeze(0)
        same_port = (port.unsqueeze(2) == port.unsqueeze(1)) & (port.unsqueeze(2) >= 0)
        adj = same_port | eye
        k = min(CFG['KNN'], N - 1)
        if k > 0:
            d2 = torch.cdist(h, h).masked_fill(~valid | eye, 1e9)
            nn_idx = d2.topk(k, dim=-1, largest=False).indices
            knn = torch.zeros(B, N, N, dtype=torch.bool, device=h.device).scatter(2, nn_idx, True)
            adj = adj | knn | knn.transpose(1, 2)
        adj = adj & (valid | eye)
        h = self.g2(self.g1(h, adj), adj)
        return h, a

class FuzzyRules(nn.Module):
    def __init__(self, d, R, K):
        super().__init__()
        self.sel = nn.Parameter(torch.randn(R, d) * 1.0 - 1.0)     # feature-selection logits
        self.theta = nn.Parameter(torch.randn(R, d) * 0.5)          # thresholds (z-score units)
        self.dirn = nn.Parameter(torch.randn(R, d))                 # direction: > (+) or < (-)
        self.beta = nn.Parameter(torch.tensor(3.0))
        self.W = nn.Parameter(torch.randn(R, K) * 0.1); self.b = nn.Parameter(torch.zeros(K))
    def forward(self, x):                                          # x [M,d]
        s = torch.sigmoid(self.sel)                                                           # [R,d]
        logmu = F.logsigmoid(self.beta * torch.tanh(self.dirn) * (x.unsqueeze(1) - self.theta))  # [M,R,d]
        r = torch.exp((s * logmu).sum(-1) / s.sum(-1).clamp(min=1e-6))   # soft AND = selection-weighted geometric mean
        return r @ self.W + self.b, r
    def l1(self): return torch.sigmoid(self.sel).mean()

class NEXUS(nn.Module):
    def __init__(self, d, K, genotype='mlp', C=128, n_rules=20, dropout=0.2, use_graph=True, use_rules=True,
                 n_layers=None, tokens=None):
        super().__init__()
        self.use_graph, self.use_rules = use_graph, use_rules
        self.enc = Encoder(d, C, tokens or CFG['TOKENS'], n_layers or CFG['N_LAYERS'], genotype, dropout)
        if use_graph: self.graph = DSTGN(C, CFG['HEADS'], dropout, CFG['SCALES'])
        self.head = nn.Sequential(nn.Linear(C, C // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(C // 2, K))
        if use_rules:
            self.rules = FuzzyRules(d, n_rules, K); self.gamma = nn.Parameter(torch.tensor(1.0))
    def forward(self, x, t, mask, port):
        B, N, d = x.shape
        h = self.enc(x.reshape(B * N, d)).reshape(B, N, -1)
        a = None
        if self.use_graph: h, a = self.graph(h, t, mask, port)
        zN = self.head(h)
        out = dict(zN=zN, scale_attn=a)
        if self.use_rules:
            zS, r = self.rules(x.reshape(B * N, d)); zS = zS.reshape(B, N, -1)
            g = torch.sigmoid(self.gamma)
            out.update(logits=g * zN + (1 - g) * zS, zS=zS, rules=r.reshape(B, N, -1), gamma=g)
        else:
            out['logits'] = zN
        return out
    def reg(self): return self.rules.l1() if self.use_rules else torch.zeros((), device=next(self.parameters()).device)

def build_model(spec, d, K):
    """spec: dict(genotype, C, n_rules, dropout, use_graph, use_rules)"""
    s = dict(genotype='mlp', C=CFG['HIDDEN'], n_rules=CFG['N_RULES'], dropout=CFG['DROPOUT'], use_graph=True, use_rules=True)
    s.update(spec)
    return NEXUS(d, K, s['genotype'], s['C'], s['n_rules'], s['dropout'], s['use_graph'], s['use_rules'],
                 n_layers=len(s['genotype']) if isinstance(s['genotype'], list) else None).to(DEVICE)

def n_params(m): return sum(p.numel() for p in m.parameters())

## Training engine and evaluation metrics

In [ ]:
def class_weights(ws, K):
    cnt = np.bincount(ws.y, minlength=K).astype(np.float64)
    w = cnt.sum() / (K * np.maximum(cnt, 1))                    # w_c = N / (K n_c)   (Eq. 1)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def node_loss(out, b, w):
    m = b['mask']
    loss = F.cross_entropy(out['logits'][m], b['y'][m], weight=w)
    if 'zS' in out: loss = loss + CFG['AUX_RULE_W'] * F.cross_entropy(out['zS'][m], b['y'][m], weight=w)
    return loss

def train_model(model, tr, va, epochs, lr=None, max_windows=None, log_batches=False, verbose=True, seed=None):
    lr = lr or CFG['LR']; rng = np.random.default_rng(seed if seed is not None else CFG['SEED'])
    if max_windows and len(tr) > max_windows: tr = tr.subset(rng.choice(len(tr), max_windows, replace=False))
    if max_windows and len(va) > max_windows: va = va.subset(rng.choice(len(va), max_windows, replace=False))
    K = model.head[-1].out_features; w = class_weights(tr, K)
    rule_p = [p for n, p in model.named_parameters() if n.startswith('rules.')]
    other_p = [p for n, p in model.named_parameters() if not n.startswith('rules.') and not n.endswith('alpha')]
    groups = [dict(params=other_p)] + ([dict(params=rule_p, lr=lr * CFG['RULE_LR_MULT'], weight_decay=0.0)] if rule_p else [])
    opt = torch.optim.Adam(groups, lr=lr, weight_decay=CFG['WD'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=1e-6)
    hist, batch_log, best, best_state, bad = [], [], 1e18, None, 0
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tl = tc = tn = 0
        for bi, b in enumerate(tr.batches(CFG['BATCH_WINDOWS'], True, rng)):
            out = model(b['x'], b['t'], b['mask'], b['port'])
            loss = node_loss(out, b, w) + CFG['RULE_L1'] * model.reg()
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            m = b['mask']; n = int(m.sum())
            tl += loss.item() * n; tn += n; tc += (out['logits'][m].argmax(-1) == b['y'][m]).sum().item()
            if log_batches and ep == 1 and bi % 10 == 0:
                batch_log.append(dict(batch=bi, loss=loss.item(), running_acc=100 * tc / tn))
        sched.step()
        vl, va_acc = evaluate_loss(model, va, w)
        hist.append(dict(epoch=ep, train_loss=tl / tn, train_acc=100 * tc / tn, val_loss=vl, val_acc=100 * va_acc,
                         seconds=time.time() - t0))
        if verbose: print(f'  epoch {ep:2d}: train loss {tl/tn:.4f} acc {100*tc/tn:.3f}% | val loss {vl:.4f} acc {100*va_acc:.3f}%')
        if vl < best - 1e-6: best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= CFG['PATIENCE']:
                if verbose: print('  early stopping'); break
    if best_state is not None: model.load_state_dict(best_state)
    return pd.DataFrame(hist), pd.DataFrame(batch_log)

@torch.no_grad()
def evaluate_loss(model, ws, w):
    model.eval(); tl = tn = tc = 0
    for b in ws.batches(4 * CFG['BATCH_WINDOWS'], False):
        out = model(b['x'], b['t'], b['mask'], b['port']); m = b['mask']; n = int(m.sum())
        tl += F.cross_entropy(out['logits'][m], b['y'][m], weight=w).item() * n; tn += n
        tc += (out['logits'][m].argmax(-1) == b['y'][m]).sum().item()
    return tl / max(tn, 1), tc / max(tn, 1)

@torch.no_grad()
def predict(model, ws, extras=False):
    model.eval(); Y, P, PR, ZS, SA, RU, IDX = [], [], [], [], [], [], []
    for b0 in range(0, len(ws), 4 * CFG['BATCH_WINDOWS']):
        widx = np.arange(b0, min(b0 + 4 * CFG['BATCH_WINDOWS'], len(ws)))
        b = ws.collate(widx); out = model(b['x'], b['t'], b['mask'], b['port']); m = b['mask']
        pr = torch.softmax(out['logits'], -1)[m]
        Y.append(b['y'][m].cpu().numpy()); PR.append(pr.cpu().numpy()); P.append(pr.argmax(-1).cpu().numpy())
        rows = np.concatenate([np.arange(s, e) for s, e in ws.win[widx]])
        IDX.append(ws.raw_index[rows])
        if extras:
            if 'zS' in out: ZS.append(out['zS'][m].argmax(-1).cpu().numpy()); RU.append(out['rules'][m].half().cpu().numpy())
            if out['scale_attn'] is not None: SA.append(out['scale_attn'][m].cpu().numpy())
    res = dict(y=np.concatenate(Y), pred=np.concatenate(P), prob=np.concatenate(PR), raw_index=np.concatenate(IDX))
    if extras:
        if ZS: res['zs_pred'] = np.concatenate(ZS); res['rules'] = np.concatenate(RU)
        if SA: res['scale_attn'] = np.concatenate(SA)
    return res

# ---------------- metrics ----------------
def conf_mat(y, p, K): return np.bincount(K * y + p, minlength=K * K).reshape(K, K)

def metrics_from_cm(cm):
    tp = np.diag(cm).astype(float); P = tp / np.maximum(cm.sum(0), 1); R = tp / np.maximum(cm.sum(1), 1)
    F1 = np.where(P + R > 0, 2 * P * R / np.maximum(P + R, 1e-12), 0); sup = cm.sum(1); w = sup / sup.sum()
    fp_b = cm[0, 1:].sum(); fn_b = cm[1:, 0].sum()
    return dict(accuracy=tp.sum() / cm.sum(), P=P, R=R, F1=F1, support=sup,
                macro_P=P.mean(), macro_R=R.mean(), macro_F1=F1.mean(),
                weighted_P=(P * w).sum(), weighted_R=(R * w).sum(), weighted_F1=(F1 * w).sum(),
                binary_FP=int(fp_b), binary_FN=int(fn_b), FPR=fp_b / max(cm[0].sum(), 1),
                FNR=fn_b / max(cm[1:].sum(), 1), errors=int(cm.sum() - tp.sum()), n=int(cm.sum()))

def summarize(res, K):
    m = metrics_from_cm(conf_mat(res['y'], res['pred'], K))
    return dict(accuracy=100 * m['accuracy'], macro_F1=100 * m['macro_F1'], weighted_F1=100 * m['weighted_F1'],
                FPR=100 * m['FPR'], FNR=100 * m['FNR'], errors=m['errors'], n=m['n'])

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def bootstrap_ci(y, p, K, B=None, seed=0):
    B = B or CFG['BOOTSTRAP']; rng = np.random.default_rng(seed); n = len(y); acc, mf1, fpr = [], [], []
    for _ in range(B):
        i = rng.integers(0, n, n); m = metrics_from_cm(conf_mat(y[i], p[i], K))
        acc.append(m['accuracy']); mf1.append(m['macro_F1']); fpr.append(m['FPR'])
    q = lambda a: (100 * np.percentile(a, 2.5), 100 * np.percentile(a, 97.5))
    return dict(accuracy=q(acc), macro_F1=q(mf1), FPR=q(fpr))

def calibration(prob, y, bins=None, B=200, seed=0):
    bins = bins or CFG['ECE_BINS']
    conf = prob.max(1); pred = prob.argmax(1); corr = (pred == y).astype(float)
    def core(conf, corr, prob, y):
        edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(conf, edges[1:-1]), 0, bins - 1)
        ece = sum(abs(corr[idx == b].mean() - conf[idx == b].mean()) * (idx == b).mean() for b in range(bins) if (idx == b).any())
        onehot = np.eye(prob.shape[1])[y]; brier = ((prob - onehot) ** 2).sum(1).mean()
        nll = -np.log(np.clip(prob[np.arange(len(y)), y], 1e-12, 1)).mean()
        lg = np.log(np.clip(conf, 1e-6, 1 - 1e-6) / np.clip(1 - conf, 1e-6, 1))
        if corr.min() == corr.max(): slope, icpt = np.nan, np.nan
        else:
            lr = LogisticRegression(C=1e6, max_iter=1000).fit(lg.reshape(-1, 1), corr)
            slope, icpt = lr.coef_[0, 0], lr.intercept_[0]
        return ece, brier, nll, slope, icpt
    point = core(conf, corr, prob, y)
    rng = np.random.default_rng(seed); boots = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y)); boots.append(core(conf[i], corr[i], prob[i], y[i]))
    boots = np.array(boots, dtype=float)
    names = ['ECE', 'Brier score', 'Negative log-likelihood', 'Calibration slope', 'Calibration intercept']
    tab = pd.DataFrame(dict(metric=names, value=point,
                            ci_low=np.nanpercentile(boots, 2.5, axis=0), ci_high=np.nanpercentile(boots, 97.5, axis=0)))
    edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(conf, edges[1:-1]), 0, bins - 1)
    rel = pd.DataFrame([dict(bin_low=edges[b], bin_high=edges[b + 1], n=int((idx == b).sum()),
                             mean_confidence=conf[idx == b].mean() if (idx == b).any() else np.nan,
                             accuracy=corr[idx == b].mean() if (idx == b).any() else np.nan) for b in range(bins)])
    return tab, rel

def threshold_table(prob, y, thresholds):
    score = 1 - prob[:, 0]; att = (y != 0); rows = []
    for th in thresholds:
        pa = score >= th; tp = int((pa & att).sum()); fp = int((pa & ~att).sum()); fn = int((~pa & att).sum()); tn = int((~pa & ~att).sum())
        rows.append(dict(threshold=th, TP=tp, FP=fp, FN=fn, TN=tn, sensitivity=tp / max(tp + fn, 1), specificity=tn / max(tn + fp, 1),
                         PPV=tp / max(tp + fp, 1), NPV=tn / max(tn + fn, 1)))
    return pd.DataFrame(rows)

def mcnemar(ya_correct, yb_correct):
    b = int((ya_correct & ~yb_correct).sum()); c = int((~ya_correct & yb_correct).sum())
    if b + c == 0: return b, c, 0.0, 1.0
    if b + c < 25: return b, c, np.nan, min(1.0, 2 * stats.binom.cdf(min(b, c), b + c, 0.5))
    chi2 = (abs(b - c) - 1) ** 2 / (b + c); return b, c, chi2, stats.chi2.sf(chi2, 1)

def holm(p):
    p = np.asarray(p); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for r, i in enumerate(o): run = max(run, min(1, (m - r) * p[i])); adj[i] = run
    return adj

def corrected_ttest(diffs, n_train, n_test):
    d = np.asarray(diffs, float); J = len(d)
    if J < 2 or d.std(ddof=1) == 0: return np.nan, np.nan
    t = d.mean() / math.sqrt((1 / J + n_test / n_train) * d.var(ddof=1))
    return t, 2 * stats.t.sf(abs(t), J - 1)

@torch.no_grad()
def latency(model, ws, n_windows=64, reps=30):
    model.eval(); b = ws.collate(np.arange(min(n_windows, len(ws)))); nodes = int(b['mask'].sum())
    for _ in range(5): model(b['x'], b['t'], b['mask'], b['port'])
    if DEVICE == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps): model(b['x'], b['t'], b['mask'], b['port'])
    if DEVICE == 'cuda': torch.cuda.synchronize()
    per_flow_batched = (time.perf_counter() - t0) / reps / nodes * 1000
    b1 = ws.collate(np.array([0]));
    for k in b1: b1[k] = b1[k][:, :1]
    for _ in range(5): model(b1['x'], b1['t'], b1['mask'], b1['port'])
    if DEVICE == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps): model(b1['x'], b1['t'], b1['mask'], b1['port'])
    if DEVICE == 'cuda': torch.cuda.synchronize()
    single = (time.perf_counter() - t0) / reps * 1000
    return dict(ms_per_flow_batched=per_flow_batched, ms_single_flow=single, batch_flows=nodes)

def fgsm_robustness(model, ws, eps, max_windows=500):
    model.eval(); w = class_weights(ws, model.head[-1].out_features); cl = ad = n = 0
    for b in ws.subset(np.arange(min(max_windows, len(ws)))).batches(CFG['BATCH_WINDOWS'], False):
        x = b['x'].clone().requires_grad_(True)
        out = model(x, b['t'], b['mask'], b['port']); m = b['mask']
        loss = F.cross_entropy(out['logits'][m], b['y'][m], weight=w); gx, = torch.autograd.grad(loss, x)
        with torch.no_grad():
            xa = x + eps * gx.sign(); oa = model(xa, b['t'], b['mask'], b['port'])
            cl += (out['logits'][m].argmax(-1) == b['y'][m]).sum().item(); ad += (oa['logits'][m].argmax(-1) == b['y'][m]).sum().item(); n += int(m.sum())
    return (ad / n) / max(cl / n, 1e-9), cl / n, ad / n

def per_class_table(res, K, names):
    m = metrics_from_cm(conf_mat(res['y'], res['pred'], K)); rows = []
    for i in range(K):
        rows.append({'class': names[i], 'support': int(m['support'][i]), 'precision_%': 100 * m['P'][i],
                     'recall_%': 100 * m['R'][i], 'F1_%': 100 * m['F1'][i]})
    for nm, key in [('Macro average', 'macro'), ('Weighted average', 'weighted')]:
        rows.append({'class': nm, 'support': int(m['n']), 'precision_%': 100 * m[key + '_P'], 'recall_%': 100 * m[key + '_R'], 'F1_%': 100 * m[key + '_F1']})
    rows.append({'class': 'Micro average (= accuracy)', 'support': int(m['n']), 'precision_%': 100 * m['accuracy'],
                 'recall_%': 100 * m['accuracy'], 'F1_%': 100 * m['accuracy']})
    return pd.DataFrame(rows)

## Control experiment — does DSTGN exploit the file order? (R4.8)
The dhoogla files have no timestamps, so graph windows are built from consecutive rows in file order.
If attack flows are stored in contiguous runs, a window can reveal the class through its neighbours.
Test: (1) measure how "pure" (single-class) the windows are; (2) randomly shuffle the row order within each
capture day (B) and across all days (C), rebuild windows and the split, and retrain. If the full model stays near its original accuracy,
the graph context is genuine; if it falls to the level of the model without DSTGN (~84 %), the gain came from
the file order.

In [ ]:
SPEC_FINAL = dict(genotype=['maxpool', 'maxpool', 'maxpool', 'dilconv3', 'conv3', 'conv5'], C=128, n_rules=20, dropout=0.1)

def window_purity(ws):
    pur = np.array([np.bincount(ws.y[s:e]).max() / (e - s) for s, e in ws.win])
    size = np.array([e - s for s, e in ws.win])
    return dict(mean_window_purity=float(pur.mean()), share_single_class_windows=float(np.mean(pur == 1)),
                mean_flows_per_window=float(size.mean()))

with Timer('Control experiment'):
    rows = []
    # A) original file order (same split as the main run)
    pA = window_purity(MAIN['test'])
    set_seed(CFG['SEED']); mA = build_model(SPEC_FINAL, N_FEAT, N_CLS)
    train_model(mA, MAIN['train'], MAIN['val'], CFG['EPOCHS'], verbose=False)
    rows.append(dict(order='original file order', config='Full NEXUS-IDS', **pA, **summarize(predict(mA, MAIN['test']), N_CLS)))
    print(rows[-1])
    # B) row order shuffled within each capture day -> windows no longer follow the file order
    ctrl = raw.copy(); rng = np.random.default_rng(0)
    ctrl['t'] = ctrl.groupby('day')['t'].transform(lambda s: rng.permutation(s.values))
    ctrl = add_time_keys(ctrl)
    SC = prepare_split(ctrl, block_split(ctrl, CFG['SEED']), 'order-shuffled control')
    pB = window_purity(SC['test'])
    for name, spec in [('Full NEXUS-IDS', SPEC_FINAL), ('w/o DSTGN', dict(SPEC_FINAL, use_graph=False))]:
        set_seed(CFG['SEED']); m = build_model(spec, len(SC['pre'].features), N_CLS)
        train_model(m, SC['train'], SC['val'], CFG['EPOCHS'], verbose=False)
        rows.append(dict(order='shuffled within day', config=name, **pB, **summarize(predict(m, SC['test']), N_CLS)))
        print(rows[-1])
    # C) shuffled across ALL days -> windows are random mixes of every day (removes day-composition cues too)
    ctrl2 = raw.copy(); ctrl2['day'] = 'all'
    ctrl2['t'] = np.random.default_rng(1).permutation(len(ctrl2)) / CFG['ROWS_PER_SEC_PROXY']
    ctrl2 = add_time_keys(ctrl2)
    SC2 = prepare_split(ctrl2, block_split(ctrl2, CFG['SEED']), 'shuffled across days')
    pC = window_purity(SC2['test'])
    for name, spec in [('Full NEXUS-IDS', SPEC_FINAL), ('w/o DSTGN', dict(SPEC_FINAL, use_graph=False))]:
        set_seed(CFG['SEED']); m = build_model(spec, len(SC2['pre'].features), N_CLS)
        train_model(m, SC2['train'], SC2['val'], CFG['EPOCHS'], verbose=False)
        rows.append(dict(order='shuffled across all days', config=name, **pC, **summarize(predict(m, SC2['test']), N_CLS)))
        print(rows[-1])
    CTRL = save_table(pd.DataFrame(rows), 'T_order_shuffle_control')

def verdict(full, nog, ref_full, ref_nog):
    gap = ref_full - ref_nog
    return (full - nog) / gap if gap > 0 else float('nan')
A_full = CTRL.iloc[0]['accuracy']
for label, i in [('shuffled within day', 1), ('shuffled across all days', 3)]:
    f, g = CTRL.iloc[i]['accuracy'], CTRL.iloc[i + 1]['accuracy']
    print(f"\n{label:26s}: full {f:.2f}% | without DSTGN {g:.2f}% | DSTGN gain {f - g:+.2f} points")
print(f"original file order       : full {A_full:.2f}%  (without DSTGN in the main ablation: about 83.8%)")
f3, g3 = CTRL.iloc[3]['accuracy'], CTRL.iloc[4]['accuracy']
share = (f3 - g3) / max(A_full - g3, 1e-9)
print(f"\nShare of the original DSTGN gain that remains when windows are random mixes of all days: {100*share:.0f}%")
print('INTERPRETATION:', 'graph context is largely genuine' if share > 0.7 else
      ('the DSTGN gain is largely an artefact of file order / day composition' if share < 0.3 else
       'the DSTGN gain is partly genuine and partly due to file order / day composition'))


▶ Control experiment ...
{'order': 'original file order', 'config': 'Full NEXUS-IDS', 'mean_window_purity': 0.9631927562643287, 'share_single_class_windows': 0.7886146496815286, 'mean_flows_per_window': 41.83479299363057, 'accuracy': np.float64(98.73345450047103), 'macro_F1': np.float64(98.73552379195442), 'weighted_F1': np.float64(98.73784137870132), 'FPR': np.float64(2.4575352367184675), 'FNR': np.float64(0.5561332911335172), 'errors': 1331, 'n': 105089}
{'order': 'shuffled within day', 'config': 'Full NEXUS-IDS', 'mean_window_purity': 0.7630890232658305, 'share_single_class_windows': 0.14685022945348353, 'mean_flows_per_window': 41.16562369628703, 'accuracy': np.float64(96.52492044510205), 'macro_F1': np.float64(88.41701900998363), 'weighted_F1': np.float64(96.75599950002592), 'FPR': np.float64(9.225211624521485), 'FNR': np.float64(0.00974342318934719), 'errors': 3429, 'n': 98674}
{'order': 'shuffled within day', 'config': 'w/o DSTGN', 'mean_window_purity': 0.7630890232658305, 'sha

## Stage 9 — Byzantine robustness (R3.2, R4.7, R5.10)
Attack model: malicious clients know their own data and the current global model, cannot see other clients'
updates, and the server is honest. Attacks on the model update Δ of a malicious client:
**sign flip** (−Δ), **Gaussian** (random update with 10× the std of Δ), **scaling / model replacement** (10·Δ),
**label flip** (local training with labels y → K−1−y). Fractions 0–40 % × FedAvg, coordinate-wise median,
Krum, trimmed mean; mean ± SD over seeds. (No DP here, to isolate the effect of the aggregator.)

In [ ]:
if CFG['RUN']['byzantine']:
    with Timer('Stage 9: Byzantine robustness'):
        rows = []
        for agg in CFG['AGGREGATORS']:
            for seed in CFG['SEEDS']:
                s, _ = run_fl(SPEC_FINAL, MAIN, PARTS_IID, CFG['BYZ_ROUNDS'], CFG['BYZ_LOCAL_EPOCHS'], None, agg, None, 0.0, seed,
                              label=f'{agg} clean', max_windows=CFG['BYZ_CLIENT_MAX_WINDOWS'])
                for attack in CFG['BYZ_ATTACKS']: rows.append(dict(s, attack=attack))
            for attack in CFG['BYZ_ATTACKS']:
                for ratio in [r for r in CFG['BYZ_RATIOS'] if r > 0]:
                    for seed in CFG['SEEDS']:
                        s, _ = run_fl(SPEC_FINAL, MAIN, PARTS_IID, CFG['BYZ_ROUNDS'], CFG['BYZ_LOCAL_EPOCHS'], None, agg, attack, ratio, seed,
                                      label=f'{agg} {attack} {ratio}', max_windows=CFG['BYZ_CLIENT_MAX_WINDOWS'])
                        rows.append(s)
                    print(f'  {agg:12s} {attack:10s} done')
        byz = pd.DataFrame(rows); save_table(byz, 'T_byzantine_all_runs', show=False)
        g = byz.groupby(['attack', 'malicious_ratio', 'aggregator']).accuracy.agg(['mean', 'std']).reset_index()
        g['acc_mean_sd'] = g.apply(lambda r: f'{r["mean"]:.3f} ± {0 if np.isnan(r["std"]) else r["std"]:.3f}', axis=1)
        piv = g.pivot_table(index=['attack', 'malicious_ratio'], columns='aggregator', values='acc_mean_sd', aggfunc='first').reset_index()
        save_table(piv, 'T_byzantine_summary')
        fig, axs = plt.subplots(1, len(CFG['BYZ_ATTACKS']), figsize=(5 * len(CFG['BYZ_ATTACKS']), 5), sharey=True)
        for ax, at in zip(np.atleast_1d(axs), CFG['BYZ_ATTACKS']):
            for agg in CFG['AGGREGATORS']:
                d = g[(g.attack == at) & (g.aggregator == agg)]; ax.errorbar(100 * d.malicious_ratio, d['mean'], yerr=d['std'].fillna(0), marker='o', label=agg, capsize=3)
            ax.set_title(at); ax.set_xlabel('Malicious clients (%)')
        np.atleast_1d(axs)[0].set_ylabel('Test accuracy (%)'); np.atleast_1d(axs)[0].legend(); save_fig(fig, 'F_byzantine', g)

## Stage 10 — Explainability: measured feature importance, rules, coverage (R3.3)
Importance of feature j = Σ_k ‖W_k‖₁ · s_kj over all rules, normalised to sum 1 (unrounded values saved).
Thresholds are converted back to the original feature units; s = feature-selection strength. A rule is shown with
its (up to 4) most selected features having s ≥ 0.5·max s. Coverage = test flows with rule activation > 0.5.

In [ ]:
if CFG['RUN']['explain'] and FINAL.use_rules:
    with Timer('Stage 10: explainability'):
        R = FINAL.rules; pre = MAIN['pre']
        s = torch.sigmoid(R.sel).detach().cpu().numpy(); th = R.theta.detach().cpu().numpy()
        dr = torch.tanh(R.dirn).detach().cpu().numpy(); W = R.W.detach().cpu().numpy()
        wnorm = np.abs(W).sum(1); imp = (wnorm[:, None] * s).sum(0); imp = imp / imp.sum()
        fi = pd.DataFrame(dict(feature=pre.features, importance=imp)).sort_values('importance', ascending=False)
        save_table(fi, 'T_feature_importance_measured')
        top = fi.head(15)[::-1]; fig, ax = plt.subplots(figsize=(10, 7)); ax.barh(top.feature, top.importance)
        ax.set_xlabel('Normalised importance (sum over all features = 1)'); save_fig(fig, 'F_feature_importance', fi)
        rows = []
        act = RES_MAIN['rules'].astype(np.float32) > 0.5
        for k in np.argsort(-wnorm):
            order_j = np.argsort(-s[k]); sel_j = [j for j in order_j[:4] if s[k, j] >= 0.5 * s[k, order_j[0]]]
            strong = s[k, order_j[0]] > 0.5
            cond = [f'{pre.features[j]} {">" if dr[k, j] > 0 else "<"} {th[k, j] * pre.std[j] + pre.mean[j]:.4g} (s={s[k, j]:.2f})' for j in sel_j]
            cls = int(W[k].argmax()); conf = float(np.exp(W[k]).max() / np.exp(W[k]).sum())
            cov = {CLASSES[c]: int((act[:, k] & (RES_MAIN['y'] == c)).sum()) for c in range(N_CLS)}
            rows.append(dict(rule=int(k), IF=' AND '.join(cond), THEN=CLASSES[cls], confidence=conf, weight_norm=float(wnorm[k]), max_selection=float(s[k].max()), selection='strong (max s > 0.5)' if strong else 'weak',
                             coverage_total=int(act[:, k].sum()), **{f'cov_{c}': v for c, v in cov.items()}))
        rules = pd.DataFrame(rows); save_table(rules, 'T_extracted_rules')
        if len(rules):
            top5 = rules.sort_values('confidence', ascending=False).groupby('THEN').head(5); save_table(top5, 'T_top5_rules_per_class')
            cc = rules.groupby('THEN')[[c for c in rules.columns if c.startswith('cov_')]].sum()
            save_table(cc.reset_index(), 'T_rule_coverage_by_class')
            fig, ax = plt.subplots(figsize=(11, 6)); cc.T.plot(kind='bar', ax=ax); ax.set_ylabel('Activated test flows'); ax.set_xlabel('True class')
            ax.set_xticklabels([c[4:] for c in cc.columns], rotation=30, ha='right'); save_fig(fig, 'F_rule_coverage', cc.reset_index())
        fid = pd.DataFrame([dict(scope='all test flows', rule_pathway_fidelity=(RES_MAIN['zs_pred'] == RES_MAIN['pred']).mean())] +
                           [dict(scope=CLASSES[c], rule_pathway_fidelity=(RES_MAIN['zs_pred'][RES_MAIN['y'] == c] == RES_MAIN['pred'][RES_MAIN['y'] == c]).mean()) for c in range(N_CLS)])
        save_table(fid, 'T_rule_fidelity'); num('fusion_gamma', float(torch.sigmoid(FINAL.gamma).item()))

## Stage 11 — Scalability with training-set size

In [ ]:
if CFG['RUN']['scalability']:
    with Timer('Stage 11: scalability'):
        rows = []; rng = np.random.default_rng(CFG['SEED'])
        for frac in CFG['SCAL_FRACS']:
            sub = MAIN['train'].subset(rng.choice(len(MAIN['train']), max(10, int(frac * len(MAIN['train']))), replace=False))
            set_seed(CFG['SEED']); m = build_model(SPEC_FINAL, N_FEAT, N_CLS)
            if DEVICE == 'cuda': torch.cuda.reset_peak_memory_stats()
            t0 = time.time(); train_model(m, sub, MAIN['val'], CFG['SCAL_EPOCHS'], verbose=False); tt = time.time() - t0
            s = summarize(predict(m, MAIN['test']), N_CLS)
            rows.append(dict(train_fraction=frac, train_flows=int(sum(e - s_ for s_, e in sub.win)),
                             train_minutes=tt / 60, test_accuracy=s['accuracy'], test_macro_F1=s['macro_F1'],
                             latency_ms_per_flow=latency(m, MAIN['test'])['ms_per_flow_batched'],
                             peak_gpu_MB=torch.cuda.max_memory_allocated() / 1e6 if DEVICE == 'cuda' else np.nan))
        sc = save_table(pd.DataFrame(rows), 'T_scalability')
        fig, ax = plt.subplots(1, 3, figsize=(18, 5))
        ax[0].plot(sc.train_flows, sc.test_accuracy, 'o-'); ax[0].set_ylabel('Test accuracy (%)')
        ax[1].plot(sc.train_flows, sc.train_minutes, 'o-'); ax[1].set_ylabel('Training time (min)')
        ax[2].plot(sc.train_flows, sc.peak_gpu_MB, 'o-'); ax[2].set_ylabel('Peak GPU memory (MB)')
        for a in ax: a.set_xscale('log'); a.set_xlabel('Training flows')
        save_fig(fig, 'F_scalability', sc)

## Stage 12 — UNSW-NB15: retrained model, **all 10 classes** reported (R4.12, R5.5)
The model is retrained on UNSW-NB15 (no transfer between datasets, because feature sets and labels differ).
The architecture found on CIC-IDS2018 is reused. This release has no timestamps or IP/port fields, so graph
windows are random groups of 32 flows and edges come from k-nearest neighbours only (stated in the paper).

In [ ]:
if CFG['RUN']['unsw']:
    with Timer('Stage 12: UNSW-NB15'):
        rd = lambda p: pd.read_parquet(p) if p.endswith('.parquet') else pd.read_csv(p)
        utr, ute = rd(CFG['UNSW_TRAIN']), rd(CFG['UNSW_TEST'])
        for d_ in (utr, ute):
            for c in d_.columns:
                if isinstance(d_[c].dtype, pd.CategoricalDtype): d_[c] = d_[c].astype(str)
            d_['attack_cat'] = d_['attack_cat'].astype(str).str.strip().replace({'Backdoors': 'Backdoor', 'nan': 'Normal', '': 'Normal', '-': 'Normal'})
        U_CLASSES = ['Normal'] + sorted([c for c in utr['attack_cat'].unique() if c != 'Normal'])
        cat_cols = [c for c in ['proto', 'service', 'state'] if c in utr.columns]
        drop = ['id', 'label', 'attack_cat']
        for c in cat_cols:
            top = utr[c].value_counts().index[:10]
            for d_ in (utr, ute): d_[c] = np.where(d_[c].isin(top), d_[c], 'other')
        allu = pd.get_dummies(pd.concat([utr, ute], keys=['tr', 'te']), columns=cat_cols, dtype=np.float32)
        U_FEATS = [c for c in allu.columns if c not in drop and pd.api.types.is_numeric_dtype(allu[c])]
        utr2, ute2 = allu.loc['tr'].reset_index(drop=True), allu.loc['te'].reset_index(drop=True)
        u_map = {c: i for i, c in enumerate(U_CLASSES)}
        for d_ in (utr2, ute2):
            d_['y'] = d_['attack_cat'].map(u_map).astype(np.int64); d_['t'] = 0.0; d_['dport'] = -1; d_['proto'] = -1; d_['day'] = 'unsw'
        itr, iva = train_test_split(np.arange(len(utr2)), test_size=0.15, stratify=utr2['y'], random_state=CFG['SEED'])
        rng = np.random.default_rng(CFG['SEED'])
        def u_ws(df, X):
            df = df.copy(); df['win'] = rng.permutation(len(df)) // 32; return make_windowset(df, X)
        upre = Preprocessor().fit(utr2.iloc[itr], U_FEATS, utr2['y'].values[itr])
        U = dict(train=u_ws(utr2.iloc[itr], upre.transform(utr2.iloc[itr])), val=u_ws(utr2.iloc[iva], upre.transform(utr2.iloc[iva])),
                 test=u_ws(ute2, upre.transform(ute2)))
        set_seed(CFG['SEED']); um = build_model(SPEC_FINAL, len(upre.features), len(U_CLASSES))
        uh, _ = train_model(um, U['train'], U['val'], CFG['UNSW_EPOCHS'])
        save_table(uh, 'T_unsw_training_history', show=False)
        ures, um_m, usum = full_evaluation(um, U, U_CLASSES, 'unsw')
        dist = pd.DataFrame(dict(cls=U_CLASSES, train=np.bincount(utr2['y'], minlength=len(U_CLASSES)), test=np.bincount(ute2['y'], minlength=len(U_CLASSES))))
        save_table(dist, 'T_unsw_class_distribution'); num('unsw', usum)
        pcu = TABLES['T_per_class_unsw'].iloc[:len(U_CLASSES)]
        big = pcu[pcu.support >= 100]
        num('unsw_macroF1_all_10_classes', float(pcu['F1_%'].mean())); num('unsw_macroF1_classes_with_100plus_test_flows', float(big['F1_%'].mean()))

## Stage 12b — CIC-IDS2017 (additional cross-dataset evaluation; optional)
Same loading, cleaning, time-block split and preprocessing as CSE-CIC-IDS2018; the model (architecture from
Stage 3) is retrained from scratch on CIC-IDS2017 and evaluated on its held-out time-block test partition.

In [ ]:
if CFG['RUN']['cic2017'] and os.path.exists(CFG['CIC2017_DIR']):
    with Timer('Stage 12b: CIC-IDS2017'):
        d17, a17 = load_flows(CFG['CIC2017_DIR']); save_table(a17, 'T_read_audit_cic2017')
        save_table(d17.groupby(['orig_label', 'cat']).size().reset_index(name='rows_after_sampling'), 'T_label_mapping_cic2017')
        d17, C17, F17 = clean_and_label(d17, 'cic2017'); d17 = add_time_keys(d17)
        p17 = block_split(d17, CFG['SEED'])
        S17 = prepare_split(d17, p17, 'cic2017', feats=F17)
        save_table(pd.crosstab(d17['cat'], p17).reindex(index=C17).reset_index().rename(columns={'cat': 'class'}), 'T_class_distribution_cic2017')
        set_seed(CFG['SEED']); m17 = build_model(SPEC_FINAL, len(S17['pre'].features), len(C17))
        h17, _ = train_model(m17, S17['train'], S17['val'], CFG['CIC2017_EPOCHS'])
        save_table(h17, 'T_cic2017_training_history', show=False)
        _, _, sum17 = full_evaluation(m17, S17, C17, 'cic2017', d17); num('cic2017', sum17)
        del d17, S17, m17; gc.collect()

## Stage 13 — Summary: one Excel workbook, key numbers, and a map from paper tables/figures to outputs

In [ ]:
PAPER_MAP = pd.DataFrame([
    ('Table 2 (class distribution)', 'T_class_distribution_splits, T_label_mapping, T_class_by_day'),
    ('Section 3.1.3 (leakage control)', 'T_read_audit (time mode), T_duplicate_audit_cic2018, T_split_leakage_audit, T_split_protocol_comparison'),
    ('Section 3.2.1 / search cost', 'T_darts_alpha_softmax, T_search_cost, T_pareto_front'),
    ('Table 3 (FL/DP configuration)', 'T_dp_accounting, T_fedids_main, T_clients_IID'),
    ('Table 4 + Figures 3–4 (per-class)', 'T_per_class_main, F_confusion_matrix_main, F_per_class_main'),
    ('Accuracy/FPR + CIs, latency, params', 'T_summary_main'),
    ('Table 5 (UNSW-NB15)', 'T_per_class_unsw, T_unsw_class_distribution, T_summary_unsw'),
    ('Additional dataset (CIC-IDS2017)', 'T_per_class_cic2017, T_summary_cic2017, F_confusion_matrix_cic2017'),
    ('Table 6 (ablation)', 'T_ablation_summary, T_ablation_all_seeds'),
    ('Table 7 + Figure 11 (McNemar)', 'T_mcnemar_all_pairs, F_mcnemar_heatmap'),
    ('Table 8 + Figure 12 right (calibration)', 'T_calibration_main, T_reliability_bins_main, F_reliability_main'),
    ('Figure 12 left (scale attention)', 'T_scale_attention_main, F_scale_attention_main'),
    ('Table 9 (cross-validation)', 'T_cv_folds, T_cv_summary, T_corrected_ttest'),
    ('Table 10 (leakage-resistant evaluation)', 'T_split_protocol_comparison, T_leave_one_day_out'),
    ('Table 11 (privacy–utility)', 'T_privacy_utility, F_privacy_utility'),
    ('Table 12 (IID vs non-IID)', 'T_fl_iid_noniid, T_fl_convergence_curves, F_fl_convergence'),
    ('Table 13 (Byzantine)', 'T_byzantine_summary, T_byzantine_all_runs, F_byzantine'),
    ('Figures 5–6 (explainability)', 'T_feature_importance_measured, T_extracted_rules, T_top5_rules_per_class, T_rule_coverage_by_class, T_rule_fidelity'),
    ('Figures 7–8 (Pareto, search stage)', 'F_pareto_search_stage, F_tradeoffs_search_stage, T_nsga_all_candidates'),
    ('Figure 2 (learning curves)', 'T_training_history, T_first_epoch_batches, F_learning_curves'),
    ('Figure 10 (scalability)', 'T_scalability, F_scalability'),
    ('Section 4.7 (errors), thresholds', 'misclassified_flows_main.csv, T_error_pairs_main, T_thresholds_main, F_threshold_sweep_main'),
    ('Appendix A (architecture)', 'T_architecture_layers, models/nexus_final_spec.json, config.json'),
], columns=['manuscript item', 'output tables/figures'])
save_table(PAPER_MAP, 'T_paper_map')
with pd.ExcelWriter(os.path.join(OUT, 'NEXUS_IDS_all_results.xlsx')) as xw:
    for name, df in TABLES.items(): df.to_excel(xw, sheet_name=name[:31], index=False)
json.dump(NUMBERS, open(os.path.join(OUT, 'paper_numbers.json'), 'w'), indent=2, default=str)
import shutil
shutil.make_archive('/kaggle/working/nexus_control_results' if OUT.startswith('/kaggle') else OUT + '_results', 'zip', OUT)
print('\nAll results written to', OUT)
print(json.dumps({k: NUMBERS[k] for k in list(NUMBERS)[:12]}, indent=1, default=str)[:3000])


=== T_paper_map ===
                        manuscript item                                                                                               output tables/figures
           Table 2 (class distribution)                                                        T_class_distribution_splits, T_label_mapping, T_class_by_day
        Section 3.1.3 (leakage control)             T_read_audit (time mode), T_duplicate_audit_cic2018, T_split_leakage_audit, T_split_protocol_comparison
            Section 3.2.1 / search cost                                                                T_darts_alpha_softmax, T_search_cost, T_pareto_front
          Table 3 (FL/DP configuration)                                                                       T_dp_accounting, T_fedids_main, T_clients_IID
      Table 4 + Figures 3–4 (per-class)                                                         T_per_class_main, F_confusion_matrix_main, F_per_class_main
    Accuracy/FPR + CIs, latency, params    